In [5]:
print("ok")

ok


In [6]:
%pwd

'c:\\Users\\OWNER\\Desktop\\python\\Medical-AI\\research'

In [33]:
import os
os.chdir("../")


In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
PINECONE_API_KEY = os.getenv("PINECONE_API_kEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")


In [6]:
%pwd

'c:\\Users\\OWNER\\Desktop\\python\\Medical-AI'

In [8]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [9]:
def load_pdf_files(data):
    loader = DirectoryLoader(
        path=data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    documents = loader.load()
    print(f"{len(documents)} documents chargés")
    return documents

In [11]:
extracted_data = load_pdf_files("Data/")

4505 documents chargés


In [ ]:
# extracted_data = load_pdf_files("Data/")
print(len(load_pdf_files("Data/")))

In [13]:
def text_splitter(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks = text_splitter.split_documents(extracted_data)
   
    return text_chunks

In [14]:
text_chunks = text_splitter(extracted_data)
print(f"{len(text_chunks)} chunks créés")

40042 chunks créés


In [15]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [16]:
#Download the Embeddings from Hugging Face
def download_hugging_face_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

In [17]:
embeddings = download_hugging_face_embeddings()

C:\Users\OWNER\AppData\Local\Temp\ipykernel_17016\979883764.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
c:\Users\OWNER\anaconda3\envs\medibot\lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [ ]:
query_result = embeddings.embed_query("Hello world")
print("length of query_result:", len(query_result))

length of query_result: 384


In [25]:
from pinecone import Pinecone

pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "sanitask"

if index_name not in [i["name"] for i in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=384,   # ⚠️ correspond à ton modèle HF
        metric="cosine",
        spec={
            "serverless": {
                "cloud": "aws",
                "region": "us-east-1"
            }
        }
    )

In [26]:
import os
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [27]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents = text_chunks,
    index_name = index_name,
    embedding = embeddings,
)

In [28]:
#Load existing index 
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding = embeddings
)

In [29]:
docsearch  

In [30]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [31]:
retrieved_docs = retriever.invoke("What is Acne?")

In [32]:
retrieved_docs

[Document(id='086807a7-c8d2-4e9e-a551-c658a49520c2', metadata={'creationdate': '2006-10-16T20:19:33+02:00', 'creator': 'Adobe Acrobat 6.0', 'moddate': '2016-02-07T11:23:03+07:00', 'page': 55.0, 'page_label': '26', 'producer': 'PDFlib+PDI 6.0.3 (SunOS)', 'source': 'Data\\Medical_book.pdf', 'total_pages': 4505.0}, page_content='Researchers, Inc. Reproduced by permission.)\n26 GALE ENCYCLOPEDIA OF MEDICINE\nAcne'),
 Document(id='3a2417f0-33c1-467f-bc28-d2390f47c715', metadata={'creationdate': '2006-10-16T20:19:33+02:00', 'creator': 'Adobe Acrobat 6.0', 'moddate': '2016-02-07T11:23:03+07:00', 'page': 269.0, 'page_label': '240', 'producer': 'PDFlib+PDI 6.0.3 (SunOS)', 'source': 'Data\\Medical_book.pdf', 'total_pages': 4505.0}, page_content='forms of acne.\nPurpose\nDifferent types of antiacne drugs are used for\ndifferent purposes. For example, lotions, soaps, gels,\nand creams containing benzoyl peroxide or tretinoin\nmay be used to clear up mild to moderately severe\nacne. Isotretinoin (A

In [ ]:
from langchain_openai import OpenAI

llm = OpenAI(temperature=0.4, max_tokens=500)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the answer concise.\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

ModuleNotFoundError: No module named 'langchain_chains'

In [ ]:

rag_chain = (
    {
        "context": retriever,
        "input": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
response = rag_chain.invoke({"input:" "what is Acne  ?"})
print(response["answer"])

In [ ]:
from langchain_aws import ChatBedrock
import os

llm = ChatBedrock(
    model_id="anthropic.claude-3-5-haiku-20241022-v1:0",
    region_name="us-east-1",
)

ValidationError: 1 validation error for ChatBedrock
  Value error, Could not load credentials to authenticate with AWS client. Please check that the specified profile name and/or its credentials are valid. Service error: You must specify a region. [type=value_error, input_value={'model_id': 'anthropic.c...se_converse_api': False}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

In [36]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

retriever = docsearch.as_retriever()

system_prompt = """
You are an assistant for question-answering tasks.
Use the retrieved context to answer the question.
If you don't know, say you don't know.
Use three sentences maximum.
Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

NameError: name 'llm' is not defined

In [ ]:
# ============================
# TEST RAG PIPELINE
# ============================

query = "What are the symptoms of diabetes?"

response = rag_chain.invoke({
    "input": query
})

print("\n🧠 QUESTION:")
print(query)

print("\n📚 RÉPONSE:")
print(response["answer"])

In [ ]:
# Ajoutez "MessagesPlaceholder" aux imports
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Modifiez la structure du prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder(variable_name="chat_history"), # L'historique sera injecté ici
    ("human", "{input}")
])